# Structured Streaming — 01: Fundamentals

## What is Structured Streaming?
Structured Streaming is a scalable, fault-tolerant stream-processing engine built on the Spark SQL engine.
It lets you express your streaming computation the same way you would express a batch computation on static data.
Internally Spark treats a **live data stream as an unbounded table** that is continuously being appended to.
Your query runs against this unbounded table, and Spark incrementally updates the result as new data arrives.

## Micro-batch Model
By default Structured Streaming uses a **micro-batch** processing model.
At each trigger interval Spark processes the new rows that arrived since the last trigger as a mini batch-job,
updates internal state (if any), and writes the result to the sink.
This gives end-to-end latencies as low as ~100 ms with exactly-once guarantees.

## Continuous vs Micro-Batch Modes
| Mode | Latency | Guarantees | When to use |
|---|---|---|---|
| Micro-batch (default) | ~100 ms–seconds | Exactly-once | Most production use-cases |
| Continuous processing | ~1 ms | At-least-once | Ultra-low latency requirements |

## Sources Covered Here
- **rate** — built-in; generates rows at a fixed rate; no external dependency
- **socket** — reads lines from a TCP socket (for quick demos)
- **memory** — reads from an in-memory table (testing only)

## Batch vs Structured Streaming API
| Aspect | Batch | Structured Streaming |
|---|---|---|
| Read | `spark.read.format(…).load()` | `spark.readStream.format(…).load()` |
| Write | `df.write.format(…).save()` | `df.writeStream.format(…).start()` |
| Blocking? | Yes | No — returns a `StreamingQuery` handle |
| `isStreaming` | `False` | `True` |
| Output modes | N/A | append / complete / update |

## Setup

In [1]:
import tempfile, time, threading
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp

spark = (
    SparkSession.builder
    .appName("SS-01-Fundamentals")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.1.2


## 1. The Rate Source — Understanding the Streaming DataFrame

The `rate` source is a built-in Spark source that **generates rows at a fixed rate**.
Each row has exactly two columns:

| Column | Type | Description |
|---|---|---|
| `timestamp` | TimestampType | Event-time of the row (wall-clock time when generated) |
| `value` | LongType | Monotonically increasing integer starting from 0 |

Because it has no external dependency it is the perfect source for local demos and unit tests.
Key options: `rowsPerSecond` (throughput), `numPartitions` (parallelism), `rampUpTime`.

In [ ]:
rate_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
)

print("isStreaming:", rate_df.isStreaming)   # True
print("\nSchema:")
rate_df.printSchema()

print("\nLogical plan (streaming):")
rate_df.explain()

isStreaming: True

Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)


Logical plan (streaming):
== Physical Plan ==
StreamingRelation rate, [timestamp#64, value#65L]




: 

## 2. writeStream — Output Modes and Triggers

### Output Modes
| Mode | Meaning | Requires aggregation? |
|---|---|---|
| `append` | Only **new** rows written to the result table since last trigger | No |
| `complete` | **Entire** result table written after each trigger | Yes (aggregation) |
| `update` | Only rows that **changed** since last trigger | No (but useful with aggregation) |

### Sink Types
| Sink | Usage |
|---|---|
| `console` | Print to stdout — dev/debug only |
| `memory` | In-memory table queryable via `spark.sql()` — testing only |
| `file` | Parquet/CSV/JSON files — production batch-style sinks |
| `foreach` / `foreachBatch` | Custom logic per row / per micro-batch |

### Trigger Modes
| Trigger | Behaviour |
|---|---|
| Default (unspecified) | Run micro-batches as fast as possible |
| `processingTime('N seconds')` | Wait N seconds between micro-batches |
| `once=True` | Process all available data as **one** batch then stop |
| `availableNow=True` | Like `once` but honours multiple micro-batches until caught up |

In [3]:
CHECKPOINT = tempfile.mkdtemp().replace('\\', '/') + '/ss01_ckpt'

query = (
    rate_df
    .writeStream
    .format("memory")
    .queryName("rate_stream")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .start()
)

print("Query started. Collecting for 6 seconds...")
time.sleep(6)

result = spark.sql("SELECT * FROM rate_stream ORDER BY value")
print(f"Rows collected: {result.count()}")
result.show(10)

query.stop()
print("Query stopped.")

Query started. Collecting for 6 seconds...
Rows collected: 0
+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+

Query stopped.


## 3. Applying Transformations on a Stream

Any DataFrame transformation that works in **batch** also works on a **streaming** DataFrame:

- `filter()` / `where()`
- `withColumn()` / `select()`
- `groupBy()` + aggregations (`count`, `sum`, `avg`, …)
- `join()` (with some restrictions on stream–stream joins)

Spark simply wraps the normal Catalyst plan in a streaming execution wrapper.
The plan is identical — only the execution engine differs (incremental vs full scan).

> **Tip:** Call `.explain()` on a streaming DataFrame to see the full logical + physical plan
> including the `StreamingRelation` node at the leaf.

In [4]:
CHECKPOINT2 = tempfile.mkdtemp().replace('\\', '/') + '/ss01_ckpt2'

transformed = (
    rate_df
    .withColumn("value_squared", col("value") * col("value"))
    .withColumn("is_even", (col("value") % 2 == 0))
    .select("timestamp", "value", "value_squared", "is_even")
)

query2 = (
    transformed
    .writeStream
    .format("memory")
    .queryName("transformed_stream")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT2)
    .start()
)

time.sleep(8)
print("Transformed stream sample:")
spark.sql("SELECT * FROM transformed_stream ORDER BY value LIMIT 12").show()
query2.stop()

Transformed stream sample:
+---------+-----+-------------+-------+
|timestamp|value|value_squared|is_even|
+---------+-----+-------------+-------+
+---------+-----+-------------+-------+



## 4. Trigger Modes — processingTime

By default Spark runs micro-batches **as fast as possible** (back-to-back with no sleep).
The `processingTime` trigger inserts a **fixed wait** between micro-batches.

```python
.trigger(processingTime="3 seconds")
```

This is useful when:
- You want to **rate-limit** processing to reduce load on downstream systems.
- You want more rows per batch (better compression, fewer small files).
- Your pipeline cost scales with number of micro-batches (e.g. cloud storage PUT costs).

If a micro-batch takes **longer** than the interval, Spark starts the next one immediately
(no backpressure needed — it self-regulates).

In [5]:
CHECKPOINT3 = tempfile.mkdtemp().replace('\\', '/') + '/ss01_ckpt3'

query3 = (
    rate_df
    .writeStream
    .format("memory")
    .queryName("trigger_demo")
    .outputMode("append")
    .trigger(processingTime="3 seconds")
    .option("checkpointLocation", CHECKPOINT3)
    .start()
)

time.sleep(10)
batches = query3.lastProgress
print("Last progress:")
import json
print(json.dumps(batches, indent=2, default=str))
query3.stop()

Last progress:
null


## 5. Query Progress and Monitoring

Every `StreamingQuery` object exposes monitoring APIs:

| API | Type | Description |
|---|---|---|
| `query.status` | dict | Current status snapshot: `isDataAvailable`, `isTriggerActive`, `message` |
| `query.lastProgress` | dict | Metrics for the most recently completed micro-batch |
| `query.recentProgress` | list[dict] | Ring-buffer of the last ~100 progress reports |
| `query.id` | UUID | Stable query ID (survives restarts from checkpoint) |
| `query.runId` | UUID | Changes every time the query is started |
| `query.name` | str | The `queryName` you set |

Key fields inside a progress dict:
- `inputRowsPerSecond` — source throughput
- `processedRowsPerSecond` — sink throughput
- `numInputRows` — rows in this micro-batch
- `batchId` — monotonically increasing micro-batch counter
- `durationMs` — breakdown of time spent in each phase

In [6]:
CHECKPOINT4 = tempfile.mkdtemp().replace('\\', '/') + '/ss01_ckpt4'

query4 = (
    rate_df
    .writeStream
    .format("memory")
    .queryName("monitoring_demo")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT4)
    .start()
)

for i in range(3):
    time.sleep(2)
    print(f'--- Poll {i+1} ---')
    print('Status:', query4.status)
    if query4.lastProgress:
        prog = query4.lastProgress
        print(f"  inputRowsPerSecond: {prog.get('inputRowsPerSecond', 'N/A')}")
        print(f"  processedRowsPerSecond: {prog.get('processedRowsPerSecond', 'N/A')}")
        print(f"  numInputRows: {prog.get('numInputRows', 'N/A')}")

query4.stop()
print("Done.")

--- Poll 1 ---
Status: {'message': "Terminated with exception: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'", 'isDataAvailable': False, 'isTriggerActive': False}
--- Poll 2 ---
Status: {'message': "Terminated with exception: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'", 'isDataAvailable': False, 'isTriggerActive': False}
--- Poll 3 ---
Status: {'message': "Terminated with exception: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'", 'isDataAvailable': False, 'isTriggerActive': False}
Done.


## 6. Trigger.Once — Exactly One Micro-Batch

`trigger(once=True)` tells Spark to:
1. Start the streaming query.
2. Process **all data currently available** in the source as a single micro-batch.
3. **Stop automatically** — no need to call `query.stop()`.

This is ideal for **scheduled incremental batch pipelines** where you want to:
- Use the streaming API (checkpoints, offset tracking, exactly-once semantics).
- But run on a schedule (cron / Airflow) rather than continuously.

Compared to a plain batch job, `Trigger.Once` gives you **offset management for free**:
the checkpoint records exactly which offsets were processed so the next run picks up where it left off.

> **Note:** `availableNow=True` (Spark 3.3+) is the successor to `once=True`.
> It processes all available data but may use multiple micro-batches for better parallelism.

In [7]:
CHECKPOINT5 = tempfile.mkdtemp().replace('\\', '/') + '/ss01_ckpt5'

# Pre-generate some data so Trigger.Once has something to process
seed_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .option("numPartitions", 1)
    .load()
)

query5 = (
    seed_df
    .writeStream
    .format("memory")
    .queryName("once_demo")
    .outputMode("append")
    .trigger(once=True)
    .option("checkpointLocation", CHECKPOINT5)
    .start()
)

query5.awaitTermination(timeout=15)
count = spark.sql("SELECT COUNT(*) as n FROM once_demo").first()["n"]
print(f"Trigger.Once processed {count} rows then stopped automatically.")

StreamingQueryException: [STREAM_FAILED] Query [id = 5b903b80-0a0f-433d-a690-2bd3ae465b08, runId = 40138c66-af7a-4202-9685-b20fefa94d8a] terminated with exception: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)' SQLSTATE: XXKST
=== Streaming Query ===
Identifier: once_demo [id = 5b903b80-0a0f-433d-a690-2bd3ae465b08, runId = 40138c66-af7a-4202-9685-b20fefa94d8a]
Current Committed Offsets: {}
Current Available Offsets: {}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSource MemorySink, 5b903b80-0a0f-433d-a690-2bd3ae465b08, [queryName=once_demo, checkpointLocation=C:/Users/PS/AppData/Local/Temp/tmpbd7kdq2_/ss01_ckpt5], Append
+- ~StreamingDataSourceV2ScanRelation[timestamp#56, value#57L] RateStream(rowsPerSecond=10, rampUpTimeSeconds=0, numPartitions=1)


## 7. Structured Streaming vs Legacy DStream

Spark originally shipped with **Spark Streaming** (aka DStream API) in Spark 1.x.
Structured Streaming replaced it in Spark 2.0 and is now the recommended approach.

| Aspect | DStream (Legacy) | Structured Streaming |
|---|---|---|
| API | RDD-based (`JavaRDD`, `PairRDD`) | DataFrame / Dataset |
| Abstraction | Sequence of RDD snapshots | Unbounded table |
| Fault tolerance | Lineage recomputation + Write-Ahead Log | Checkpoint + offset log |
| Event time | Manual (complex) | Built-in `window()`, watermarks |
| Late data | Not handled | Handled via watermarks |
| Exactly-once | Hard to guarantee end-to-end | Supported with idempotent sinks |
| SQL support | None | Full Spark SQL |
| Status | Maintenance mode (no new features) | Active development |

**Bottom line:** always use Structured Streaming for new projects.

## Key Takeaways

- A streaming DataFrame looks exactly like a batch DataFrame but `isStreaming` is `True`.
- The `rate` source is the simplest way to get a local streaming DataFrame with no external deps.
- Choose **output mode** based on your query type:
  - `append` — raw events, no aggregation
  - `complete` — running totals / full aggregation result
  - `update` — only rows that changed (works with or without aggregation)
- Use the **memory sink** during development so you can `spark.sql()` the results.
- Monitor live queries with `.status`, `.lastProgress`, and `.recentProgress`.
- `Trigger.Once` is a powerful pattern for bridging streaming and scheduled batch pipelines.

## Practice Exercises

1. Modify the rate source to emit **5 rows/second** and collect for **10 seconds**.
   How many rows do you get? Is it exactly 50? Why or why not?

2. Add a filter `col("value") > 10` before writing to the memory sink.
   Does it work on streams? What does the explain plan look like?

3. Change `outputMode` to `"complete"` on a raw (non-aggregated) stream.
   What `AnalysisException` do you get and why?

4. Use `query.recentProgress` to print the **last 3 progress reports** for a query
   that has been running for 15 seconds with `processingTime("2 seconds")`.

## Teardown

In [ ]:
spark.stop()
print("SparkSession stopped.")